# Tutorial SAM 2.1: segmentasi gambar dan video
Notebook pendamping blog *Dari “Klik Objeknya” ke “Sebut Namanya”: Perjalanan SAM 2 menuju SAM 3* oleh Abdul Haris (Advanced Computer Vision, S2 TI UNPAM).

**Sebelum mulai:** Runtime → Change runtime type → **T4 GPU**. Jalankan sel dari atas ke bawah.
Semua output di notebook ini kosong; hasil akan muncul saat kamu menjalankannya sendiri.

## 1. Instal SAM 2 dan unduh checkpoint (tiny, ±156 MB)

In [ ]:
!pip install -q "git+https://github.com/facebookresearch/sam2.git"
!mkdir -p checkpoints
!wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
!wget -q -O truck.jpg https://raw.githubusercontent.com/facebookresearch/sam2/main/notebooks/images/truck.jpg
!wget -q -O bedroom.zip https://dl.fbaipublicfiles.com/segment_anything_2/assets/bedroom.zip
!unzip -q -o bedroom.zip

Catatan: pesan error saat membangun ekstensi CUDA opsional boleh diabaikan (sesuai README resmi).

## 2. Impor dan pilih perangkat

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from sam2.build_sam import build_sam2, build_sam2_video_predictor
from sam2.sam2_image_predictor import SAM2ImagePredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Perangkat:", device, "| torch", torch.__version__)

## 3. Muat model

In [ ]:
checkpoint = "checkpoints/sam2.1_hiera_tiny.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_t.yaml"

predictor = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint, device=device))

## 4. Segmentasi gambar dengan satu klik
Gambar contoh adalah `truck.jpg` dari repositori resmi. Ganti dengan gambarmu sendiri
(unggah lewat panel Files di kiri) dan sesuaikan koordinat klik `(x, y)`.

In [ ]:
image = np.array(Image.open("truck.jpg").convert("RGB"))
point = np.array([[500, 375]])      # (x, y) piksel yang diklik
label = np.array([1])               # 1 = objek, 0 = latar

predictor.set_image(image)
masks, scores, _ = predictor.predict(
    point_coords=point, point_labels=label, multimask_output=True
)
order = np.argsort(scores)[::-1]
masks, scores = masks[order], scores[order]
print("Skor tiga kandidat mask:", np.round(scores, 3))

In [ ]:
def overlay(ax, mask, color=(0.12, 0.56, 1.0, 0.6)):
    h, w = mask.shape[-2:]
    ax.imshow(mask.reshape(h, w, 1) * np.array(color).reshape(1, 1, -1))

fig, axes = plt.subplots(1, len(masks), figsize=(5 * len(masks), 5))
for ax, m, s in zip(axes, masks, scores):
    ax.imshow(image)
    overlay(ax, m)
    ax.scatter(*point[0], c="lime", marker="*", s=300, edgecolor="white")
    ax.set_title(f"skor {s:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

**Latihan:** tambahkan klik negatif (label `0`) di area yang salah masuk mask, lalu panggil `predict` lagi
dengan `multimask_output=False`. Bandingkan hasilnya.

## 5. Segmentasi dan pelacakan video
Secara default notebook memakai klip contoh `bedroom` (±200 frame JPEG) dari aset resmi SAM 2, yang sudah diunduh di sel instal.

Untuk videomu sendiri, ubah jadi frame JPEG lalu ganti `VIDEO_DIR`:
```
!mkdir -p frames
!ffmpeg -i video.mp4 -q:v 2 -start_number 0 frames/%05d.jpg
```
Nama file harus berupa angka urut (`00000.jpg`, `00001.jpg`, ...). Gunakan klip pendek (beberapa detik) agar memori GPU cukup,
dan sesuaikan koordinat kotak dengan objek di videomu.

In [ ]:
import os
VIDEO_DIR = "bedroom"   # ganti ke "frames" untuk videomu sendiri
assert os.path.isdir(VIDEO_DIR), f"Folder {VIDEO_DIR} tidak ada. Jalankan sel instal atau siapkan frame JPEG dulu."

video_predictor = build_sam2_video_predictor(model_cfg, checkpoint, device=device)

state = video_predictor.init_state(video_path=VIDEO_DIR)

# Prompt di frame 0: kotak (x1, y1, x2, y2) di sekeliling anak perempuan pada klip bedroom (960x540).
# Satu klik saja sering hanya menangkap SEBAGIAN objek (mis. celana atau kaki); kotak memberi cakupan yang utuh.
video_predictor.add_new_points_or_box(
    inference_state=state,
    frame_idx=0,
    obj_id=1,
    box=np.array([340, 10, 540, 470], dtype=np.float32),
)

video_masks = {}
for frame_idx, obj_ids, mask_logits in video_predictor.propagate_in_video(state):
    video_masks[frame_idx] = (mask_logits[0] > 0.0).cpu().numpy()
print("Jumlah frame yang berhasil dilacak:", len(video_masks))

In [ ]:
frame_names = sorted(os.listdir(VIDEO_DIR))
shown = list(video_masks)[::max(1, len(video_masks) // 5)][:5]

fig, axes = plt.subplots(1, len(shown), figsize=(4 * len(shown), 4))
axes = np.atleast_1d(axes)
for ax, idx in zip(axes, shown):
    ax.imshow(Image.open(os.path.join(VIDEO_DIR, frame_names[idx])))
    overlay(ax, video_masks[idx])
    ax.set_title(f"frame {idx}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Eksperimen untuk laporanmu
1. Bandingkan `tiny` dan `large` (checkpoint `sam2.1_hiera_large.pt`, config `configs/sam2.1/sam2.1_hiera_l.yaml`): waktu inferensi dan kualitas mask.
2. Uji pada objek tipis atau transparan. Di mana model gagal?
3. Ganti prompt kotak dengan satu klik positif, mis. `points=np.array([[400, 270]], dtype=np.float32), labels=np.array([1], dtype=np.int32)`, dan lihat bagaimana mask hanya menangkap sebagian tubuh. Lalu tambahkan klik lain untuk memperbaikinya.
4. Pada video, tambahkan prompt koreksi di frame tengah dengan `add_new_points_or_box` (frame_idx lain), lalu jalankan ulang `propagate_in_video`.
5. Catat hasilnya dan jadikan bahan analisis di blogmu.

Sumber: <https://github.com/facebookresearch/sam2>